# Модель данных сертификата

Ветка: **[ДПО ИИ] 10 — Сертификат или удостоверение**.

Этот ноутбук является частью будущей Jupyter Book-книги и предназначен для воспроизводимой демонстрации модели.

## 1. Загрузка структурированных данных

Сущности и связи сертификата вынесены в YAML, чтобы таблицы не терялись при переносе в инженерную среду.

In [ ]:
from pathlib import Path
import yaml, json
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

BASE = Path.cwd()
entities_path = BASE / 'data' / 'certificate_entities.yaml'
with entities_path.open(encoding='utf-8') as f:
    spec = yaml.safe_load(f)
list(spec['entities'].keys())

## 2. Таблица сущностей

Таблица показывает назначение ключевых сущностей: шаблон документа, выданный документ, объект хранения и аудит.

In [ ]:
rows = []
for entity_name, entity in spec['entities'].items():
    rows.append({
        'entity': entity_name,
        'description': entity['description'],
        'fields_count': len(entity['fields']),
        'relationships_count': len(entity.get('relationships', []))
    })
entities_df = pd.DataFrame(rows)
entities_df

## 3. Поля IssuedCertificate

`IssuedCertificate` — центральная сущность этапа. Она связывает слушателя, запуск курса, зачисление, шаблон, PDF-файл и код проверки.

In [ ]:
issued_fields = pd.DataFrame(spec['entities']['IssuedCertificate']['fields'])
issued_fields

## 4. Граф связей сущностей

Граф показывает, что сертификат не является изолированным PDF-файлом. Он связан с пользователем, зачислением, запуском курса, организацией, шаблоном, хранилищем и аудитом.

In [ ]:
G = nx.DiGraph()
for rel in spec['relationships']:
    G.add_edge(rel['source'], rel['target'], label=rel['label'])

plt.figure(figsize=(12, 7))
pos = nx.spring_layout(G, seed=42, k=1.2)
nx.draw_networkx_nodes(G, pos, node_size=2400)
nx.draw_networkx_edges(G, pos, arrows=True, arrowstyle='-|>', arrowsize=16)
nx.draw_networkx_labels(G, pos, font_size=9)
edge_labels = nx.get_edge_attributes(G, 'label')
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)
plt.title('Связи сущностей в модели сертификата')
plt.axis('off')
output = BASE / 'assets' / 'diagrams' / 'certificate_data_model_graph.png'
plt.savefig(output, dpi=200, bbox_inches='tight')
plt.show()
output

## 5. Пример записи выданного сертификата

Пример демонстрирует полный служебный объект `IssuedCertificate` и отдельный публичный ответ проверки. Это важно: публичная страница не должна раскрывать внутренние идентификаторы, хеши, email, служебные ключи хранения и метаданные подписи.

In [ ]:
payload_path = BASE / 'data' / 'example_certificate_payload.json'
with payload_path.open(encoding='utf-8') as f:
    payload = json.load(f)

payload['issuedCertificate']

In [ ]:
pd.DataFrame([
    {'field': k, 'value_type': type(v).__name__, 'public': k in ['registrationNumber','issuedAt','hours','competencies','status','verificationUrl']}
    for k, v in payload['issuedCertificate'].items()
])

## 6. Вывод

Модель документа должна хранить полный служебный контекст в защищённом контуре, но публичная проверка должна возвращать только минимально необходимую информацию. Это снижает риск раскрытия персональных данных и сохраняет проверяемость документа.